# US Company Valuation Pipeline
## 미국 기업 밸류에이션 분석 파이프라인

이 노트북은 미국 기업들의 매출 예측 및 밸류에이션을 계산합니다.
- 4개 예측 모델 사용: SARIMA, LSTM, Prophet, Exponential Smoothing
- PSR(Price-to-Sales Ratio) 기반 밸류에이션
- DB 자동 저장

## 1. 라이브러리 및 모듈 임포트

In [1]:
# -*- coding: utf-8 -*-

import sys
import os
import warnings
import argparse
import gc
import traceback
import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')

## 2. 프로젝트 경로 설정 및 모듈 로드

In [2]:
# 프로젝트 경로 설정
from utils import add_repo_path, log, to_month_end_safe

project_root = add_repo_path()
print(f"Project root: {project_root}")

Project root: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [3]:
# 예측 모델 임포트 및 리로드
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)

import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)

import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)

import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

from DATA.stock_invest_function import get_db_host

print("✅ 예측 모델 모듈 로드 완료")


✅ 예측 모델 모듈 로드 완료


In [4]:
# 커스텀 모듈 임포트
from config import api_key, db_config, BATCH_SIZE, start_date_month, get_default_month_end_str
from db_audit import audit_db_coverage
from data_fetch import (fetch_revenue_data, fetch_market_data_yearly, fetch_db_revenue_data,
                        safe_get_db_market_df, process_daily_to_monthly_market_data)
from data_processing import clean_rev_data, calculate_enhanced_ttm_and_psr, prepare_revenue_ttm
from forecast_models import run_revenue_forecasts, run_psr_forecasts, pick_columns
from valuation import calculate_valuation, flush_batch_and_upload
from db_batch_save import save_batch_results_to_db

print("✅ 커스텀 모듈 로드 완료")

✅ 커스텀 모듈 로드 완료


## 3. 설정 및 초기화

In [5]:
# DB 설정
db_config['host'] = get_db_host()
db_info = db_config

print("DB 설정:")
print(f"  Host: {db_info['host']}")
print(f"  Port: {db_info['port']}")
print(f"  Database: {db_info['database']}")
print(f"  User: {db_info['user']}")

DB 설정:
  Host: 192.168.0.230
  Port: 3307
  Database: investar
  User: stox7412


In [6]:
# 날짜 설정
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

print(f"분석 기간:")
print(f"  시작: {start_date_month}")
print(f"  종료: {end_date_month}")

분석 기간:
  시작: 2011-03-01
  종료: 2025-10-31


In [24]:
# Ticker 리스트 설정
# from DATA.us_target_ticker_list import ticker_list

# ★★★ 테스트용: AAPL만 실행 ★★★
target_tickers = ['AAPL']

# 전체 실행할 때는 아래 주석 해제
# target_tickers = ticker_list[0:100]  # 원하는 범위 지정

print(f"처리할 Ticker 수: {len(target_tickers)}")
print(f"Ticker 리스트: {target_tickers[:10]}...")  # 처음 10개만 표시

처리할 Ticker 수: 1
Ticker 리스트: ['AAPL']...


In [25]:
# 전역 변수 초기화
error_ticker_list = []
total_success_tickers = 0
total_upsert_rows = 0
total_revenue_rows = 0

batch_results = []
batch_revenue_results = []

print("✅ 변수 초기화 완료")

✅ 변수 초기화 완료


## 4. DB 커버리지 감사

In [26]:
# DB에서 데이터 존재 여부 확인
miss_q, miss_m = audit_db_coverage(db_info, target_tickers)

if miss_m:
    fname = f"market_cap_missing_ticker_{pd.Timestamp.today().strftime('%Y%m%d')}.csv"
    pd.DataFrame({"ticker": miss_m}).to_csv(fname, index=False, encoding="utf-8-sig")
    log("AUDIT-SAVE", f"US_fundm missing {len(miss_m)} tickers saved -> {fname}")
else:
    log("AUDIT-SAVE", "US_fundm missing tickers: 0")

[00:29:24] AUDIT: US_fundq missing: 0 tickers
[00:29:24] AUDIT: US_fundm missing: 0 tickers
[00:29:24] AUDIT-SAVE: US_fundm missing tickers: 0


## 5. 개별 Ticker 처리 함수 정의

In [27]:
ticker ='AAPL'

from get_fmp_revenue import get_fmp_revenue
from get_fmp_marketcap_processor import get_fmp_market_cap

# revenue_data, error = fetch_revenue_data(ticker, api_key)

rev_data, error = get_fmp_revenue(
    ticker=ticker,
    api_key=api_key,
    fetch_revenue_data_func=fetch_revenue_data,  # 실제 함수
    fetch_db_revenue_func=fetch_db_revenue_data,  # 실제 함수
    db_info=db_info,
    log_func=log,
    error_list=error_ticker_list
)

# 4. 매출 예측
sarima_df, lstm_df, prophet_raw_df, es_raw_df, error = run_revenue_forecasts(
    rev_data, ticker, sarima, lstm_v2, prophet_v3, esmod,
)
if error:
    error_ticker_list.append({'ticker': ticker, 'stage': 'revenue_forecast', 'error': error})


# 2. 기존 코드 (5-6번 블록) 전체를 아래 한 줄로 교체
merged_market_df, error = get_fmp_market_cap(
    ticker=ticker,
    api_key=api_key,
    fetch_market_data_func=fetch_market_data_yearly,
    process_daily_to_monthly_func=process_daily_to_monthly_market_data,
    fetch_db_market_func=safe_get_db_market_df,
    db_info=db_info,
    start_year=2010,
    log_func=log,
    error_list=error_ticker_list
)

# Import 추가
from get_marketcap_fundamental_dataframe import get_marketcap_fundamental_dataframe

# 기존 코드를 이 한 줄로 교체
market_cap_resize = get_marketcap_fundamental_dataframe(
    merged_market_df=merged_market_df,
    rev_data=rev_data,
    start_date = start_date_month,
    end_date = end_date_month,
    log_func=log
)

# Import 추가
from get_marketcap_fundamental_dataframe import get_marketcap_fundamental_dataframe

# 기존 코드를 이 한 줄로 교체
market_cap_resize = get_marketcap_fundamental_dataframe(
    merged_market_df=merged_market_df,
    rev_data=rev_data,
    start_date = start_date_month,
    end_date = end_date_month,
    log_func=log
)

enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)
psr_ok = enhanced_merged_df_with_ttm[['date_month_end', 'PSR_ttm']].dropna()

# 8. PSR 예측
psr_sarima_df, psr_lstm_df, psr_prophet_df, psr_es_df, error = run_psr_forecasts(
    enhanced_merged_df_with_ttm, ticker, sarima, lstm_v2, prophet_v3, esmod
)

rev_sarima = pick_columns(sarima_df, ['revenue_billions_sarima_noexog'])
rev_lstm = pick_columns(lstm_df, ['revenue_billions_lstm_forecast'])
rev_prophet = pick_columns(prophet_raw_df, ['revenue_billions_prophet_forecast'])
rev_es = pick_columns(es_raw_df, ['revenue_billions_esq_forecast'])

revenue_forecast_df = pd.concat([rev_sarima, rev_lstm, rev_prophet, rev_es], axis=1, join='outer')
# revenue_forecast_df 저장 필요

# Revenue forecast 데이터를 배치에 추가
revenue_forecast_for_db = revenue_forecast_df.reset_index()
revenue_forecast_for_db['ticker'] = ticker
batch_revenue_results.append(revenue_forecast_for_db)
log("OK-REV-BATCH", f"{ticker} added to revenue batch, batch_size={len(batch_revenue_results)}")
#
# # TTM 계산
revenue_forecast_df_reset = revenue_forecast_df.reset_index()
if 'ticker' not in revenue_forecast_df_reset.columns:
    revenue_forecast_df_reset['ticker'] = ticker
#
revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df_reset)
revenue_forecast_ttm = revenue_forecast_.filter(like='_ttm').copy()
#
if 'ticker' not in revenue_forecast_df_reset.columns or revenue_forecast_df_reset['ticker'].isna().all():
    raise ValueError("ticker 칼럼이 필요합니다.")

if 'date_month_end' not in revenue_forecast_.columns:
    revenue_forecast_ttm['date_month_end'] = revenue_forecast_df_reset['date_month_end'].values
else:
    revenue_forecast_ttm['date_month_end'] = revenue_forecast_['date_month_end'].values
revenue_forecast_ttm = revenue_forecast_ttm.drop_duplicates(subset=['date_month_end']).set_index(
    'date_month_end')

revenue_cols_ttm = [
    'revenue_billions_sarima_noexog_ttm',
    'revenue_billions_lstm_forecast_ttm',
    'revenue_billions_prophet_forecast_ttm',
    'revenue_billions_esq_forecast_ttm'
]
revenue_forecast_ttm['revenue_billions_avg_of_4_ttm'] = revenue_forecast_ttm[revenue_cols_ttm].mean(axis=1)

psr_sarima = pick_columns(psr_sarima_df, ['PSR_ttm_sarima_forecast'])
print(psr_sarima)
psr_lstm = pick_columns(psr_lstm_df, ['PSR_ttm_lstm_forecast'])
psr_prophet = pick_columns(psr_prophet_df, ['PSR_prophet_forecast_noexog'])
psr_es = pick_columns(psr_es_df, ['PSR_es_forecast'])
psr_forecast_df = pd.concat([psr_sarima, psr_lstm, psr_prophet, psr_es], axis=1, join='outer')

valuation_result = calculate_valuation(revenue_forecast_ttm, psr_forecast_df, ticker)
batch_results.append(valuation_result)

[00:29:25] FMP-REV-REQ: AAPL url=https://financialmodelingprep.com/api/v3/income-statement/AAPL limit=200 period=quarter
[00:29:26] FMP-REV-RESP: AAPL status=200
[00:29:26] FMP-REV-DATA: AAPL type=list size=159
[00:29:26] OK-FMP-REV: AAPL raw_rows=159
[00:29:26] CHK-FMP-MEND: AAPL nan_mend=0 / 159
[00:29:27] DB-REV-OK: AAPL rows=554 range=2000-01-31~2025-07-31
[00:29:27] OK-DB-REV: AAPL rows=554
[00:29:27] REV-MERGE: AAPL merged_rows=47
[00:29:27] OK-REV-CLEAN: AAPL rows=47 yrmin=2015 yrmax=2025


00:29:40 - cmdstanpy - INFO - Chain [1] start processing
00:29:41 - cmdstanpy - INFO - Chain [1] done processing


[00:29:41] OK-REV-FORECAST: AAPL sarima=(53, 6) lstm=(50, 6) prophet=(51, 6) es=(47, 2)
[00:29:41] FMP-MCAP-REQ: AAPL year=2010
[00:29:41] FMP-MCAP-RESP: AAPL year=2010 status=200
[00:29:42] FMP-MCAP-REQ: AAPL year=2011
[00:29:42] FMP-MCAP-RESP: AAPL year=2011 status=200
[00:29:57] FMP-MCAP-REQ: AAPL year=2025
[00:29:58] FMP-MCAP-RESP: AAPL year=2025 status=200
[00:29:58] FMP-MCAP-DONE: AAPL total_records=3983
[00:29:58] OK-FMP-MCAP: AAPL rows=190
[00:29:58] DB-MCAP-REQ: AAPL
[00:29:58] DB-MCAP-OK: AAPL rows=316 range=2000-01-31~2025-04-30
[00:29:58] DB-MCAP-RESP: AAPL rows=316
[00:29:58] OK-MCAP-MERGE: AAPL rows=190 nan_mcap=0
[00:29:58] OK-MERGE: Final rows=128 range=2011-03-01 to 2025-10-31
[00:29:58] OK-MERGE: Final rows=128 range=2011-03-01 to 2025-10-31


00:30:27 - cmdstanpy - INFO - Chain [1] start processing
00:30:27 - cmdstanpy - INFO - Chain [1] done processing


[INFO] 예측 시작일: 2025-11-30 | 데이터 마지막 월: 2025-10-31
[00:30:27] OK-PSR-FORECAST: AAPL sarima=(146, 2) lstm=(146, 8) prophet=(146, 8) es=(146, 8)
[00:30:27] OK-REV-BATCH: AAPL added to revenue batch, batch_size=1
                PSR_ttm_sarima_forecast
date_month_end                         
2015-03-31                          NaN
2015-04-30                          NaN
2015-05-31                    12.868299
2015-06-30                     6.132219
2015-07-31                     3.953859
...                                 ...
2026-12-31                    10.285012
2027-01-31                     9.403349
2027-02-28                     8.439414
2027-03-31                     7.653047
2027-04-30                     7.197612

[146 rows x 1 columns]


In [36]:
revenue_forecast_df

,revenue_billions_sarima_noexog,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_esq_forecast
date_month_end,,,,
2015-03-31,58.010000,58.010000,58.010000,58.010000
2015-06-30,49.600000,49.600000,49.600000,49.600000
2015-09-30,51.500000,51.500000,51.500000,51.500000
2015-12-31,75.870000,75.870000,75.870000,75.870000
2016-03-31,50.560000,50.560000,50.560000,50.560000
2016-06-30,42.360000,42.360000,42.360000,42.360000
2016-09-30,46.850000,46.850000,46.850000,46.850000
2016-12-31,78.350000,78.350000,78.350000,78.350000
2017-03-31,52.896000,52.896000,52.896000,52.896000


## 6. Ticker별 처리 및 배치 저장

In [11]:
# 배치 크기 설정
batch_size = 20

# Ticker별 처리
for idx, ticker in enumerate(target_tickers, 1):
    success = process_ticker(
        ticker, idx, len(target_tickers), api_key, db_info, start_date_month, end_date_month,
        batch_results, batch_revenue_results, error_ticker_list
    )

    if success:
        total_success_tickers += 1

    # 배치 플러시
    try:
        is_last = (idx == len(target_tickers))
        if (len(batch_results) >= batch_size) or is_last:
            log("BATCH-FLUSH",
                f"valuation={len(batch_results)}, revenue={len(batch_revenue_results)}, is_last={is_last}")

            # ★★★ batch_results를 Long format으로 변환하여 DB 저장 ★★★
            if batch_results:
                psr_rows = save_batch_results_to_db(
                    batch_results,
                    db_info,
                    forecast_date=pd.Timestamp.today().strftime('%Y-%m-%d'),
                    table_name="us_psr_valuation_result"
                )
                log("PSR-VALUATION-SAVE", f"Saved {psr_rows} rows to us_psr_valuation_result")

            total_upsert_rows, total_revenue_rows = flush_batch_and_upload(
                batch_results, batch_revenue_results, db_info, total_upsert_rows, total_revenue_rows
            )
    except Exception as e:
        log("EXC-BATCH-FLUSH", f"{ticker} e={e}")
        error_ticker_list.append({'ticker': ticker, 'stage': 'batch_flush', 'error': str(e)})

    # 메모리 정리
    gc.collect()

[11:52:00] TICKER: 1/1 AAPL
[11:52:00] FMP-REV-REQ: AAPL url=https://financialmodelingprep.com/api/v3/income-statement/AAPL limit=200 period=quarter
[11:52:01] FMP-REV-RESP: AAPL status=200
[11:52:01] FMP-REV-DATA: AAPL type=list size=159
[11:52:01] OK-FMP-REV: AAPL raw_rows=159
[11:52:01] CHK-FMP-MEND: AAPL nan_mend=0 / 159
[11:52:04] DB-REV-OK: AAPL rows=554 range=2000-01-31~2025-07-31
[11:52:04] OK-DB-REV: AAPL rows=554
[11:52:04] REV-MERGE: AAPL merged_rows=59
[11:52:04] OK-REV-CLEAN: AAPL rows=57 yrmin=2011 yrmax=2025





11:52:19 - cmdstanpy - INFO - Chain [1] start processing
11:52:19 - cmdstanpy - INFO - Chain [1] done processing


[11:52:20] OK-REV-FORECAST: AAPL sarima=(61, 9) lstm=(61, 9) prophet=(61, 9) es=(61, 2)
[11:52:20] FMP-MCAP-REQ: AAPL year=2010
[11:52:20] FMP-MCAP-RESP: AAPL year=2010 status=200
[11:52:21] FMP-MCAP-REQ: AAPL year=2011
[11:52:21] FMP-MCAP-RESP: AAPL year=2011 status=200
[11:52:36] FMP-MCAP-REQ: AAPL year=2025
[11:52:36] FMP-MCAP-RESP: AAPL year=2025 status=200
[11:52:37] FMP-MCAP-DONE: AAPL total_records=3983
[11:52:37] OK-FMP-MCAP: AAPL rows=190
[11:52:37] DB-MCAP-REQ: AAPL
[11:52:37] DB-MCAP-OK: AAPL rows=316 range=2000-01-31~2025-04-30
[11:52:37] DB-MCAP-RESP: AAPL rows=316
[11:52:37] OK-MCAP-MERGE: AAPL rows=190 nan_mcap=0
[11:52:37] OK-PSR-PREP: AAPL rows=170


11:53:16 - cmdstanpy - INFO - Chain [1] start processing


[INFO] 예측 시작일: 2025-11-30 | 데이터 마지막 월: 2025-10-31


11:53:16 - cmdstanpy - INFO - Chain [1] done processing


[11:53:16] OK-PSR-FORECAST: AAPL sarima=(182, 2) lstm=(182, 8) prophet=(182, 8) es=(182, 8)
[11:53:16] OK-REV-BATCH: AAPL added to revenue batch, batch_size=1
[11:53:16] OK-VAL-PACK: AAPL packed=15 batch=1
[11:53:16] BATCH-FLUSH: valuation=1, revenue=1, is_last=True
[11:53:16] DB-TABLE: Table us_psr_valuation_result ensured with utf8mb4_unicode_ci
[11:53:16] LONG-FORMAT: Converted 15 wide rows to 195 long rows
[11:53:17] DB-UPSERT: Upserted 195 rows to us_psr_valuation_result
[11:53:17] SAVE-BATCH: Successfully saved 195 rows to us_psr_valuation_result
[11:53:17] PSR-VALUATION-SAVE: Saved 195 rows to us_psr_valuation_result
[11:53:17] BATCH-UPLOADED: valuation rows=10, total=10
[11:53:17] BATCH-REV-CHECK: revenue batch contains tickers: ['AAPL']
[11:53:17] DB-REV-UPSERT: AAPL upserted 61 rows
[11:53:17] BATCH-UPLOADED: revenue forecast rows=61, total=61


## 7. 결과 확인 및 저장

In [ ]:
# 최종 결과 출력
print("\n" + "="*80)
print("처리 완료")
print("="*80)
print(f"성공 ticker: {total_success_tickers}")
print(f"Valuation 업서트 rows: {total_upsert_rows}")
print(f"Revenue forecast 업서트 rows: {total_revenue_rows}")
print(f"오류 ticker: {len(error_ticker_list)}")

In [ ]:
# 오류 리스트 저장
if error_ticker_list:
    try:
        error_df = pd.DataFrame(error_ticker_list)
        error_df.to_csv("valuation_error_list.csv", index=False, encoding="utf-8-sig")
        print(f"\n오류 리스트 저장: valuation_error_list.csv (총 {len(error_ticker_list)}개)")
        display(error_df)
    except Exception as e:
        print(f"오류 리스트 저장 실패: {e}")
else:
    print("\n✅ 오류 없이 완료")

## 8. 결과 데이터 확인 (선택사항)

In [9]:
# DB에서 저장된 데이터 확인
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
)

# PSR Valuation 결과 확인
query = """
SELECT * FROM us_psr_valuation_result
WHERE ticker = ''
ORDER BY date DESC, indicator
LIMIT 50
"""

result_df = pd.read_sql(query, engine)
print(f"\n조회된 행 수: {len(result_df)}")
display(result_df)


조회된 행 수: 0


,id,date,ticker,indicator,value,forecast_date,created_ts,updated_ts


In [10]:
# Indicator별 개수 확인
query = """
SELECT indicator, COUNT(*) as count
FROM us_psr_valuation_result
WHERE ticker = 'SMCK'
GROUP BY indicator
ORDER BY indicator
"""

indicator_count = pd.read_sql(query, engine)
print("\nIndicator별 데이터 개수:")
display(indicator_count)


Indicator별 데이터 개수:


,indicator,count
